# PlaSim emulator — skill figures

Figures for slides: PhysicsNeMo emulator vs the SFNO **v11** baseline, on held-out
PlaSim sim52 years 122–131.

**Kernel:** use the `aires` conda env — it has matplotlib/numpy. The notebook only
reads JSON, so it needs neither torch nor makani.

```
/work2/11079/aasch/stampede3/conda-envs/aires/bin/python -m ipykernel install \
    --user --name aires --display-name "Python (aires)"
```

**Colours** are the Okabe–Ito palette, the standard colour-vision-deficiency-safe set
for scientific figures (validated here: worst adjacent ΔE 9.6 under deuteranopia,
20.0 under normal vision). Every series also carries a distinct **line style**, so the
figures stay readable in greyscale and when printed.

Every figure cell ends with `save(fig, "name")`, which writes a 200-dpi PNG and a
vector PDF into `figures/` — drop either straight into slides.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

RES = "/work2/11079/aasch/stampede3/runs/plasim_pnemo"
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

# ---- Okabe-Ito, colour-vision-deficiency safe -------------------------------
OKABE = {
    "blue":    "#0072B2",
    "vermil":  "#D55E00",
    "green":   "#009E73",
    "orange":  "#E69F00",
    "purple":  "#CC79A7",
    "skyblue": "#56B4E9",
    "black":   "#000000",
    "yellow":  "#F0E442",
}

# Colour AND line style per model: style is a second encoding, so the figures
# survive greyscale printing and any form of colour blindness.
STYLE = {
    "v11":    dict(color=OKABE["black"],   ls=(0, (5, 3)), label="SFNO v11 (baseline)"),
    "stage1": dict(color=OKABE["skyblue"], ls=(0, (1, 1.5)), label="R1 s1 · single-step"),
    "stage2": dict(color=OKABE["orange"],  ls=(0, (3, 1, 1, 1)), label="R1 s2 · unroll 2"),
    "stage3": dict(color=OKABE["purple"],  ls=(0, (4, 1.5)), label="R1 s3 · unroll 4"),
    "stage4": dict(color=OKABE["vermil"],  ls="-", label="R1 s4 · ensemble"),
    "r2":     dict(color=OKABE["blue"],    ls="-", label="R2 · rollout to 8 d"),
}
ORDER = ["v11", "stage1", "stage2", "stage3", "stage4", "r2"]

# ---- slide-friendly defaults ------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13,
    "legend.fontsize": 11, "xtick.labelsize": 12, "ytick.labelsize": 12,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.4, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

def save(fig, name):
    """Write a 200-dpi PNG and a vector PDF for slides."""
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT}/{name}.{ext}")
    print(f"saved {OUT}/{name}.png and .pdf")

# ---- no-skill references (measured, see Round 2 of the plan) ---------------
CLIM_PRECIP = 5.108   # mm/day - RMSE of predicting local climatology
CLIM_T2M    = 7.006   # K
print("matplotlib", plt.matplotlib.__version__)

## Load results

`skill_comparison.json` holds v11 and Round 1 stages 1–4; `skill_r2_only.json` holds
Round 2, computed later with the identical script, so they merge cleanly.

In [ ]:
d = json.load(open(f"{RES}/skill_comparison.json"))
d["models"]["r2"] = json.load(open(f"{RES}/skill_r2_only.json"))["models"]["r2"]
LEAD_D = np.array(d["lead_hours"]) / 24.0          # 6-hourly, out to 15 days
M = d["models"]

# Round 2 daily-mean diagnostics
diag4 = json.load(open(f"{RES}/diagnose_stage4_cpu.json"))
diagr = json.load(open(f"{RES}/diagnose_r2_unroll32.json"))
tail  = json.load(open(f"{RES}/daily_tail_compare.json"))
spread = json.load(open(f"{RES}/spread_diag.json"))
DAYS = np.array(diag4["days"])

print("models:", list(M))
print("leads:", len(LEAD_D), "to", LEAD_D[-1], "days | daily diagnostics:", len(DAYS), "days")

## Figure 1 — 2 m temperature

The clean comparison: `tas` is a prognostic state channel and v11 shares our units and
climatology (its `tas` mean 277.87 K vs our 277.6), so nothing is rescaled.

**Slide caption:** *Round 2 beats the v11 baseline at every lead, and overtakes Round 1's
best model from day 5 onward.*

In [ ]:
def lead_panel(ax, metric, models=ORDER, xmax=15, ylab=""):
    for k in models:
        ax.plot(LEAD_D, M[k][metric], **STYLE[k])
    ax.set_xlim(0, xmax); ax.set_xlabel("lead time (days)"); ax.set_ylabel(ylab)
    ax.xaxis.set_major_locator(MultipleLocator(3))
    return ax

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
lead_panel(axes[0], "tas_rmse_global",  ylab="RMSE (K)").set_title("2 m temperature — global")
lead_panel(axes[1], "tas_rmse_Chicago", ylab="RMSE (K)").set_title("2 m temperature — Chicago")
for ax in axes:
    ax.axhline(CLIM_T2M, color="0.55", lw=1.2, ls=":", zorder=0)
    ax.annotate("climatology (no skill)", xy=(0.4, CLIM_T2M), xytext=(0.4, CLIM_T2M*0.93),
                fontsize=10, color="0.4")
axes[0].legend(ncol=2, loc="upper left")
fig.suptitle("Round 2 beats v11 at every lead", y=1.02, fontsize=16, fontweight="semibold")
save(fig, "fig1_t2m_rmse"); plt.show()

## Figure 2 — precipitation, 6-hourly

**Caveat to state on the slide:** v11's precipitation channel is *not* our field in
different units — its climatological mean ratio (3496) and std ratio (4331) disagree and
its zero fraction is 12% against our 31%, because its postprocessing interpolates. It is
**z-score matched** onto our distribution before scoring, so its line measures **pattern
skill, not absolute error**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
lead_panel(axes[0], "pr_rmse_global",        ylab="RMSE (mm/day)").set_title("Precipitation — global")
lead_panel(axes[1], "pr_rmse_Bay_of_Bengal", ylab="RMSE (mm/day)").set_title("Precipitation — Bay of Bengal")
axes[0].axhline(CLIM_PRECIP, color="0.55", lw=1.2, ls=":", zorder=0)
axes[0].annotate("climatology (no skill)", xy=(0.4, CLIM_PRECIP),
                 xytext=(0.4, CLIM_PRECIP*1.04), fontsize=10, color="0.4")
axes[0].legend(ncol=2, loc="lower right")
fig.suptitle("Precipitation: v11 line is pattern skill only (see note)",
             y=1.02, fontsize=16, fontweight="semibold")
save(fig, "fig2_precip_rmse"); plt.show()

## Figure 3 — skill score, and where each model stops beating climatology

Probably the most slide-worthy figure. Skill = `1 − RMSE / RMSE_climatology`; the zero
line is where a model stops being better than simply predicting the local climatology.

**Slide caption:** *Round 2 pushes the no-skill crossover from day 7 to beyond day 10.*

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.2))
cross = {}
for k in ORDER:
    sk = 1 - np.array(M[k]["pr_rmse_global"]) / CLIM_PRECIP
    ax.plot(LEAD_D, 100*sk, **STYLE[k])
    below = np.where(sk <= 0)[0]
    cross[k] = LEAD_D[below[0]] if len(below) else None
ax.axhline(0, color="0.3", lw=1.4)
ax.set_xlim(0, 15); ax.set_ylim(-60, 80)
ax.set_xlabel("lead time (days)"); ax.set_ylabel("skill vs climatology (%)")
ax.set_title("Global precipitation skill; below 0 = worse than climatology")
ax.xaxis.set_major_locator(MultipleLocator(3))
ax.legend(ncol=2, loc="upper right")
save(fig, "fig3_precip_skill_score"); plt.show()

print("no-skill crossover (days):")
for k, v in cross.items():
    print(f"  {k:8s} {'never within 15 d' if v is None else f'{v:.2f}'}")

## Figure 4 — daily-mean precipitation, and *why* skill decays

The dashed lines feed the model the **true state** at each lead instead of its own
forecast. They are flat — so given a perfect state the precipitation head is excellent and
stays excellent. **All** of the decay is state error propagating into the diagnosis, not a
failure of the precipitation head.

**Slide caption:** *The precipitation head was never the bottleneck — the state is.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
for nm, dd, key in [("stage4", diag4, "stage4"), ("r2", diagr, "r2")]:
    st = STYLE[key]
    ax.plot(DAYS, dd["crps_pred"], color=st["color"], ls="-", label=f"{st['label']} — forecast state")
    ax.plot(DAYS, dd["crps_true"], color=st["color"], ls=(0, (2, 2)), lw=2.0,
            label=f"{st['label']} — TRUE state")
ax.set_xlabel("lead time (days)"); ax.set_ylabel("daily-mean CRPS (mm/day)")
ax.set_title("Given a perfect state, the head is flat"); ax.legend(fontsize=9.5)

ax = axes[1]
ax.plot(DAYS, diag4["csi"], color=STYLE["stage4"]["color"], ls="-", label="stage 4")
ax.plot(DAYS, diagr["csi"], color=STYLE["r2"]["color"], ls="-", label="round 2")
ax.set_xlabel("lead time (days)"); ax.set_ylabel("critical success index")
ax.set_title("Wet/dry decision (higher is better)"); ax.legend()
fig.suptitle("Round 2 cuts daily-mean CRPS 31–34% at days 3–10",
             y=1.02, fontsize=16, fontweight="semibold")
save(fig, "fig4_daily_crps_attribution"); plt.show()

## Figure 5 — the two defects Round 3 must fix

Measured on **single ensemble members**, not the ensemble mean. That distinction matters:
an earlier version of this analysis compared the *ensemble mean* to truth and concluded the
model was over-smoothing. That was wrong — a calibrated ensemble mean is legitimately
smoother than any single field, so the falling ratio was expected, not a defect.

Corrected, the picture reverses and sharpens:

* members carry **too much** small-scale power (r2 ≈ 2× truth), because the precipitation
  sampler draws every grid point independently — a member is the smooth mean field plus
  spatially uncorrelated noise;
* members **lack large coherent events** (p99 16.96 vs 20.99 observed);
* both models are **badly under-dispersed** (spread/skill 0.33–0.44, where ~1.0 is calibrated).

**Slide caption:** *Better scores, but individual forecasts are not yet realistic.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
sd = {k: spread[k] for k in ("stage4", "r2")}

ax = axes[0]
for k in ("stage4", "r2"):
    ax.plot(DAYS, sd[k]["member_spec"], color=STYLE[k]["color"], ls="-", label=STYLE[k]["label"])
ax.axhline(1.0, color="0.3", lw=1.4, ls=":")
ax.annotate("truth", xy=(1, 1.0), xytext=(1, 1.06), fontsize=10, color="0.35")
ax.set_ylabel("power ratio, single member"); ax.set_xlabel("lead time (days)")
ax.set_title("Small-scale power (1.0 = realistic)"); ax.legend(fontsize=10)

ax = axes[1]
ax.plot(DAYS, sd["stage4"]["q99_obs"], color="0.35", ls=":", lw=2.0, label="observed")
for k in ("stage4", "r2"):
    ax.plot(DAYS, sd[k]["q99_member"], color=STYLE[k]["color"], ls="-", label=STYLE[k]["label"])
ax.set_ylabel("p99 of daily precip (mm/day)"); ax.set_xlabel("lead time (days)")
ax.set_title("Extremes in a single forecast"); ax.legend(fontsize=10)

ax = axes[2]
for k in ("stage4", "r2"):
    ax.plot(DAYS, sd[k]["spread_skill"], color=STYLE[k]["color"], ls="-", label=STYLE[k]["label"])
ax.axhline(1.0, color="0.3", lw=1.4, ls=":")
ax.annotate("calibrated", xy=(1, 1.0), xytext=(1, 1.03), fontsize=10, color="0.35")
ax.set_ylim(0, 1.2)
ax.set_ylabel("spread / skill"); ax.set_xlabel("lead time (days)")
ax.set_title("Ensemble dispersion (1.0 = calibrated)"); ax.legend(fontsize=10)

fig.suptitle("What Round 3 needs to fix", y=1.03, fontsize=16, fontweight="semibold")
save(fig, "fig5_round3_targets"); plt.show()

## Numbers for the slide text

Prints the headline values so you can paste exact figures into a slide rather than
reading them off a chart.

In [ ]:
def at(days_, arr, day):
    return arr[int(np.argmin(np.abs(np.array(days_) - day)))]

print("Global RMSE at day 1 / 5 / 15")
print(f"{'model':9s} {'t2m (K)':>22s}   {'precip (mm/day)':>24s}")
for k in ORDER:
    t = [at(LEAD_D, M[k]["tas_rmse_global"], d) for d in (1, 5, 15)]
    p = [at(LEAD_D, M[k]["pr_rmse_global"], d) for d in (1, 5, 15)]
    print(f"{k:9s} " + " ".join(f"{v:6.3f}" for v in t) + "   " + " ".join(f"{v:7.2f}" for v in p))

print("\nDaily-mean precip CRPS (mm/day)")
print(f"{'day':>4s} {'stage4':>8s} {'round2':>8s} {'change':>8s}")
for d in (1, 3, 5, 7, 10):
    a = at(DAYS, diag4["crps_pred"], d); b = at(DAYS, diagr["crps_pred"], d)
    print(f"{d:4d} {a:8.3f} {b:8.3f} {100*(b-a)/a:+7.0f}%")

print("\nRound 3 targets at day 10")
for k in ("stage4", "r2"):
    s = spread[k]
    print(f"  {k:7s} member power {s['member_spec'][-1]:.2f} (want 1.0) | "
          f"p99 {s['q99_member'][-1]:.1f} vs obs {s['q99_obs'][-1]:.1f} | "
          f"spread/skill {s['spread_skill'][-1]:.2f} (want ~1.0)")

## Export everything

Re-runs every figure and writes it to `figures/`. PNG for slides, PDF if you want vector.

**Caveats worth carrying onto the slides:**

1. v11's precipitation is **z-score matched** — its precip lines are pattern skill, not
   absolute error. Its state (t2m) comparison needs no such caveat.
2. v11 is deterministic, so any "CRPS" for it is **MAE** — the correct reduction, but not
   the same quantity as an ensemble CRPS.
3. Figure 5 uses **single members**; the ensemble-mean version of that metric is
   misleading, for the reason given above.

In [ ]:
import glob
print("figures written:")
for f in sorted(glob.glob(f"{OUT}/*")):
    print("  ", f, f"({os.path.getsize(f)//1024} KB)")